In [1]:
# !pip install requests psycopg2-binary python-dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

API_KEY = os.getenv("OPENWEATHER_API_KEY")

print("API Loaded:", API_KEY is not None)

API Loaded: True


In [2]:
CITIES = ["Hatton", "Galle"]   # change anytime
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

In [3]:
import logging
from datetime import datetime
import os

os.makedirs("../logs", exist_ok=True)

log_file = f"../logs/pipeline_{datetime.now().date()}.log"

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Pipeline started")

In [4]:
import requests
import time

def fetch_weather(city, retries=3):
    
    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    for attempt in range(retries):
        try:
            response = requests.get(BASE_URL, params=params, timeout=10)

            if response.status_code == 200:
                logging.info(f"API success: {city}")
                return response.json()

            else:
                logging.warning(f"API failed ({city}) - Status {response.status_code}")

        except Exception as e:
            logging.error(f"Error fetching {city}: {e}")

        time.sleep(2)

    logging.error(f"API failed after retries: {city}")
    return None

In [5]:
data = fetch_weather("Colombo")
data

{'coord': {'lon': 79.8478, 'lat': 6.9319},
 'weather': [{'id': 801,
   'main': 'Clouds',
   'description': 'few clouds',
   'icon': '02d'}],
 'base': 'stations',
 'main': {'temp': 31.13,
  'feels_like': 35.91,
  'temp_min': 30.97,
  'temp_max': 31.13,
  'pressure': 1011,
  'humidity': 63,
  'sea_level': 1011,
  'grnd_level': 1010},
 'visibility': 10000,
 'wind': {'speed': 4.32, 'deg': 254, 'gust': 3.81},
 'clouds': {'all': 24},
 'dt': 1772781225,
 'sys': {'type': 2,
  'id': 2103021,
  'country': 'LK',
  'sunrise': 1772758295,
  'sunset': 1772801560},
 'timezone': 19800,
 'id': 1248991,
 'name': 'Colombo',
 'cod': 200}

In [6]:
import json
from datetime import datetime
import os

def save_raw_data(city, data):

    os.makedirs("../data/raw", exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"../data/raw/{city}_{timestamp}.json"

    with open(filename, "w") as f:
        json.dump(data, f)

    logging.info(f"Raw data saved: {filename}")

In [7]:
save_raw_data("Colombo", data)

In [8]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="de_lab",
    user="postgres",
    password="root"
)

cur = conn.cursor()

print("Connected successfully")

Connected successfully


In [9]:
create_table_query = """
CREATE TABLE IF NOT EXISTS weather_raw (
    id SERIAL PRIMARY KEY,
    city TEXT,
    api_response JSONB,
    ingestion_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

cur.execute(create_table_query)
conn.commit()

print("Table created")

Table created


In [10]:
def insert_raw(city, data):

    insert_query = """
    INSERT INTO weather_raw (city, api_response)
    VALUES (%s, %s);
    """

    cur.execute(insert_query, (city, json.dumps(data)))
    conn.commit()

    logging.info(f"Inserted raw data: {city}")

In [11]:
insert_raw("Colombo", data)

In [12]:
data.keys()

dict_keys(['coord', 'weather', 'base', 'main', 'visibility', 'wind', 'clouds', 'dt', 'sys', 'timezone', 'id', 'name', 'cod'])

In [13]:
from datetime import datetime

def transform_weather_data(raw_json):

    transformed = {
        "city": raw_json["name"],
        "temperature": raw_json["main"]["temp"],
        "humidity": raw_json["main"]["humidity"],
        "pressure": raw_json["main"]["pressure"],
        "wind_speed": raw_json["wind"]["speed"],
        "weather_desc": raw_json["weather"][0]["description"],
        "observation_time": datetime.fromtimestamp(raw_json["dt"]),
        "ingestion_time": datetime.now()
    }

    return transformed

In [14]:
clean_data = transform_weather_data(data)
clean_data

{'city': 'Colombo',
 'temperature': 31.13,
 'humidity': 63,
 'pressure': 1011,
 'wind_speed': 4.32,
 'weather_desc': 'few clouds',
 'observation_time': datetime.datetime(2026, 3, 6, 12, 43, 45),
 'ingestion_time': datetime.datetime(2026, 3, 6, 12, 47, 30, 53169)}

In [15]:
create_clean_table = """
CREATE TABLE IF NOT EXISTS weather_clean (
    id SERIAL PRIMARY KEY,
    city TEXT,
    temperature FLOAT,
    humidity INT,
    pressure INT,
    wind_speed FLOAT,
    weather_desc TEXT,
    observation_time TIMESTAMP,
    ingestion_time TIMESTAMP
);
"""

cur.execute(create_clean_table)
conn.commit()

print("Clean table created")

Clean table created


In [16]:
def insert_clean(data):

    insert_query = """
    INSERT INTO weather_clean (
        city,
        temperature,
        humidity,
        pressure,
        wind_speed,
        weather_desc,
        observation_time,
        ingestion_time
    )
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s);
    """

    cur.execute(insert_query, (
        data["city"],
        data["temperature"],
        data["humidity"],
        data["pressure"],
        data["wind_speed"],
        data["weather_desc"],
        data["observation_time"],
        data["ingestion_time"]
    ))

    conn.commit()
    logging.info(f"Inserted clean data: {data['city']}")

In [17]:
insert_clean(clean_data)

In [18]:
def run_pipeline():

    logging.info("Pipeline execution started")

    for city in CITIES:

        raw = fetch_weather(city)

        if raw is None:
            continue

        save_raw_data(city, raw)
        insert_raw(city, raw)

        clean = transform_weather_data(raw)
        insert_clean(clean)

    logging.info("Pipeline execution finished")

In [19]:
run_pipeline()